# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mayurkharche01/Internship-starter-flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane is **Content Opportunity Scoring**, which is a scoring task.  
The goal is to give each content item a priority score so the team can decide which pages to review first.

In [5]:
import pandas as pd

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Unique clients: {df['client_id'].nunique():,}")

Rows: 30,000
Columns: 44
Unique content items: 30,000
Unique clients: 32


## 2. Target or proxy

The dataset does not contain the actual result of a content refresh, so I use an observed proxy.  
The proxy identifies content items that are currently showing a downward trend in performance.

In [6]:
print("Available columns:")
print(df.columns.tolist())

# Check that the required column exists
if "trend_direction" not in df.columns:
    raise KeyError("Column 'trend_direction' was not found in the dataset.")

df["decline_proxy"] = (
    df["trend_direction"].astype(str).str.lower().str.strip() == "down"
).astype(int)

print("\nProxy distribution:")
print(df["decline_proxy"].value_counts().sort_index())

print("\nProxy rate:")
print(f"{df['decline_proxy'].mean() * 100:.1f}% of content items are currently trending down.")

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Proxy distribution:
decline_proxy
0    13738
1    16262
Name: count, dtype: int64

Proxy rate:
54.2% of content items are currently trending down.


## 3. Success metric

I will use **Precision@K** to measure how useful the top-ranked content items are.  
It shows how many of the selected top-K items match the observed proxy outcome.

In [7]:
K = 100

# Simple demonstration using the observed proxy.
# This is not an ML model and is only used to show how Precision@K works.
top_k = df.nlargest(K, "impressions_90d")

precision_at_k = top_k["decline_proxy"].mean()

print(f"K = {K}")
print(f"Proxy-positive items in top K: {top_k['decline_proxy'].sum():,}")
print(f"Precision@{K}: {precision_at_k:.3f}")

K = 100
Proxy-positive items in top K: 38
Precision@100: 0.380


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one content item**.  
Each row represents one content item with information such as traffic, engagement, age, and recent performance.

In [8]:

display_cols = [
    "content_id",
    "client_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "engagement_rate",
    "impressions_last_30d",
    "impressions_prev_30d",
    "decline_proxy"
]

lane_df = df[display_cols].copy()

print(f"Rows shown: {len(lane_df):,}")
print("One row = one content item")

display(lane_df.head(10))

Rows shown: 30,000
One row = one content item


,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,engagement_rate,impressions_last_30d,impressions_prev_30d,decline_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,29,17,5.88,578,987,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,7,9,0.00,2501,5915,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,11,11,0.00,2382,6089,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,58,78,1.28,3626,4206,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,24,145,0.00,4211,6452,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,1,5,0.00,617,1009,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,20,0,1,0.00,1,13,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,1,28,3.57,636,632,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,32574,29,68,5.88,5696,13828,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,1240,2,3,0.00,252,356,1


## 5. Why ML beats a fixed rule here

A fixed rule may consider only one or two signals, while content priority depends on several factors together.  
ML can combine these signals to create a useful priority score, but it should first be compared with a simple rule.

In [9]:
signal_cols = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "engagement_rate",
    "impressions_last_30d",
    "impressions_prev_30d"
]

print("Candidate signals available for later modeling:")
print("\n".join(f"- {col}" for col in signal_cols))

print(f"\nNumber of candidate signals listed: {len(signal_cols)}")

Candidate signals available for later modeling:
- content_age_days
- days_since_last_update
- impressions_90d
- clicks_90d
- sessions_90d
- engagement_rate
- impressions_last_30d
- impressions_prev_30d

Number of candidate signals listed: 8


## Self-check

Before submission:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, proxy, directional, decision-support
- [x] The task is framed around a real content-review decision
- [x] The unit of analysis is shown as a real dataframe
- [x] A success metric is named before model training
- [x] I have not used `trend_direction` or `trend_pct` as model features
- [ ] Committed to my repo under `work/notebooks/`
- [ ] Submitted the repo URL## Self-check

